In [1]:
!pip install fastapi uvicorn pyngrok transformers accelerate bitsandbytes nest_asyncio
!pip -q install flash-attn --no-build-isolation



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 75.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
!nvidia-smi

Wed Dec 17 13:20:15 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-40GB          Off |   00000000:00:04.0 Off |                    0 |
| N/A   29C    P0             43W /  400W |       0MiB /  40960MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

In [1]:
from huggingface_hub import login

# You will be prompted to paste your HF token (make a token at: https://huggingface.co/settings/tokens)
login()


In [6]:
from fastapi import FastAPI, Query
from pydantic import BaseModel
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re
import time
from typing import List, Dict, Any, Tuple, Optional

# =========================================================
# MODEL LOADING (LLAMA, FASTEST RELIABLE CONFIG)
# =========================================================

MODEL_NAME = "meta-llama/Meta-Llama-3-8B-Instruct"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def load_model():
    assert torch.cuda.is_available(), "GPU required (A100 expected)"

    # Speed optimizations for A100
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print("Loading model (BF16 + FlashAttention2)...")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        torch_dtype=torch.bfloat16,     # faster than 4-bit on A100
        device_map={"": 0},
        attn_implementation="flash_attention_2",
    )
    model.eval()
    return model

model = load_model()
DEVICE = "cuda:0"
print(f"Using device: {DEVICE}")

# =========================================================
# LABELS
# =========================================================

LABELS = [
    "INVENTION",
    "COMPONENT",
    "SUBSYSTEM",
    "MATERIAL",
    "CHEMICAL",
    "BIOMOLECULE",
    "COMPOSITION",
    "PROCESS_STEP",
    "METHOD",
    "PARAMETER",
    "MEASUREMENT",
    "CONDITION",
    "FUNCTION",
    "SIGNAL",
    "CONTROL",
    "SOFTWARE",
    "HARDWARE",
    "FIGURE_REF",
    "CLAIM_ELEMENT",
    "PRIOR_ART",
    "UNCLASSIFIED_ENTITY",
]

# =========================================================
# FASTAPI APP
# =========================================================

app = FastAPI()

class PredictRequest(BaseModel):
    data: list  # [{"text": "..."}]

# =========================================================
# HELPERS
# =========================================================

def _extract_json_array(s: str) -> List[Dict[str, Any]]:
    m = re.search(r"\[\s*{.*?}\s*\]", s, flags=re.DOTALL)
    if not m:
        m = re.search(r"\[.*\]", s, flags=re.DOTALL)
    if not m:
        return []
    try:
        parsed = json.loads(m.group(0))
        return parsed if isinstance(parsed, list) else []
    except Exception:
        return []

def _find_span(text: str, entity_text: str, start_from: int = 0) -> Tuple[int, int]:
    idx = text.find(entity_text, start_from)
    if idx == -1:
        return (-1, -1)
    return (idx, idx + len(entity_text))

# =========================================================
# CORE NER LOGIC (LLAMA, NO EARLY STOP)
# =========================================================

def extract_spans(text: str) -> List[Dict[str, Any]]:
    labels_str = ", ".join(LABELS)

    prompt = f"""
You are a patent NER engine.

Extract span entities from the input text.
Use ONLY these labels:
{labels_str}

Rules:
- Return ONLY a valid JSON array.
- Each item must be an object with keys: "text" and "label".
- "text" MUST be copied VERBATIM from the input text.
- Do NOT paraphrase or shorten.
- If unsure, OMIT the entity.
- Do NOT create overlapping spans.

Input text:
\"\"\"{text}\"\"\"

Return JSON array now:
""".strip()

    inputs = tokenizer(prompt, return_tensors="pt").to(DEVICE)

    gen_kwargs = dict(
        max_new_tokens=80,        # ✅ as requested
        do_sample=False,
        temperature=0.0,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    t0 = time.time()
    with torch.no_grad():
        output = model.generate(**inputs, **gen_kwargs)
    t1 = time.time()

    prompt_len = inputs["input_ids"].shape[-1]
    gen_ids = output[0][prompt_len:]
    decoded = tokenizer.decode(gen_ids, skip_special_tokens=True)

    dt = max(t1 - t0, 1e-6)
    print(f"[NER] {gen_ids.shape[-1]} tokens in {dt:.2f}s", flush=True)

    items = _extract_json_array(decoded)

    spans = []
    used_ranges = []

    for it in items:
        ent_text = str(it.get("text", "")).strip()
        label = str(it.get("label", "")).strip()

        if not ent_text or label not in LABELS:
            continue

        start, end = _find_span(text, ent_text)
        if start == -1:
            continue

        if any(not (end <= s or start >= e) for s, e in used_ranges):
            continue

        used_ranges.append((start, end))
        spans.append(
            {
                "start": start,
                "end": end,
                "text": text[start:end],
                "labels": [label],
            }
        )

    return spans

# =========================================================
# ROUTES
# =========================================================

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE, "model": MODEL_NAME}

@app.api_route("/setup", methods=["GET", "POST"])
def setup():
    return {
        "from_name": "label",
        "to_name": "text",
        "type": "labels",
        "labels": LABELS,
    }

@app.api_route("/predict", methods=["GET", "POST"])
def predict(
    request: Optional[PredictRequest] = None,
    text: Optional[str] = Query(default=None),
):
    if request is not None and request.data:
        data = request.data
    elif text is not None:
        data = [{"text": text}]
    else:
        return []

    print(f"🔵 /predict received: n_items={len(data)}", flush=True)

    results = []
    for item in data:
        t = item.get("text", "")
        if not t:
            continue

        spans = extract_spans(t)

        results.append(
            {
                "result": [
                    {
                        "from_name": "ner",
                        "to_name": "text",
                        "type": "labels",
                        "value": span,
                    }
                    for span in spans
                ],
                "score": 1.0,
            }
        )

    return results


Loading tokenizer...


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

Loading model (BF16 + FlashAttention2)...


...

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Using device: cuda:0


In [7]:
import uvicorn, threading, nest_asyncio, time, requests

nest_asyncio.apply()

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")

# start server in background
server_thread = threading.Thread(target=run_server, daemon=True)
server_thread.start()

time.sleep(5)  # give it a few seconds to start

# quick test: local health
print(requests.get("http://127.0.0.1:8000/health").json())


INFO:     Started server process [11424]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): [errno 98] address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     127.0.0.1:37040 - "GET /health HTTP/1.1" 200 OK
{'status': 'ok', 'device': 'cuda:0', 'model': 'meta-llama/Meta-Llama-3-8B-Instruct'}


In [8]:
from google.colab import output

public_url = output.eval_js("google.colab.kernel.proxyPort(8000)")
public_url


'https://8000-gpu-a100-s-3ilu1aphsdoti-f.us-central1-1.prod.colab.dev'

In [10]:
from pyngrok import ngrok
from dotenv import load_dotenv
load_dotenv()
import os
# kill old tunnels in this session
ngrok.kill()

# your auth token
ngrok.set_auth_token("2PgsprcdKolcczw6ru6HXbLcYfC_7cUSXnTdho7wqZyHYotoF")

# A) random domain
# public_url = ngrok.connect(addr="127.0.0.1:8000")

# B) your reserved free domain
public_url = ngrok.connect(
    addr="127.0.0.1:8000",
    domain="empiristic-mariyah-unprophetically.ngrok-free.dev"
)

print("Public URL:", public_url)


Public URL: NgrokTunnel: "https://empiristic-mariyah-unprophetically.ngrok-free.dev" -> "http://127.0.0.1:8000"


InvalidSchema: No connection adapters were found for '127.0.0.1:8000/api/predictions/bulk/'